<a href="https://colab.research.google.com/github/tiffanysoe1/Reinforcement-Learning/blob/main/ReinforcementLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reinforcement Learning Prototype


---

Create a custom 1D environment called LineWorld, verify it against a random action baseline and trains PPO agent, packages the files for Docker Deployment, and wraps the environment in a live FastAPI web service.

1. Environment Setup & Dependencies
2. Custom Environment Definition (`env.py`)
3. Environment Verification & Random Baseline
4. PPO Model Training
5. Dockerization & Project Export
6. REST API Server (`server.py`)
7. Local API Smoke Test

In [ ]:
!pip install -q gymnasium==1.0.0 stable-baselines3

In [ ]:
%%writefile env.py
import gymnasium as gym
from gymnasium import spaces

class LineWorld(gym.Env):
  def __init__(self, size=10):
    self.size = size
    self.observation_space = spaces.Discrete(self.size)
    self.action_space = spaces.Discrete(2) #O is left, #1 is right

  def reset(self, seed=None, options=None):
    super().reset(seed=seed)
    self.pos = 0
    return self.pos, {}

  def step(self, action):
    self.pos = max(0, min(self.size - 1, self.pos + (1 if action == 1 else -1 )))
    terminated = self.pos == self.size - 1
    reward = 1.0 if terminated else -0.01
    return self.pos, reward, terminated, False, {}


Overwriting env.py


In [ ]:
#Test with random agent
import importlib
import env
from env import LineWorld
from gymnasium.utils.env_checker import check_env

env = LineWorld()
check_env(env)

obs, _ = env.reset(seed=0)
done, total = False, 0
while not done:
  obs, r, done, _, _ = env.step(env.action_space.sample())
  total += r
print("random agent return:", total)


random agent return: -2.5799999999999677


In [ ]:
!pip install -q stable-baselines3

In [ ]:
from stable_baselines3 import PPO
import os

os.makedirs("output", exist_ok=True)
model = PPO("MlpPolicy", LineWorld(), verbose=0)
model.learn(total_timesteps=20_000)
model.save("output/model")


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
!pip freeze | grep -E "gymnasium|stable.baselines3|numpy" > requirements.txt

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
CMD ["python", "train.py"]

Writing Dockerfile


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!zip -r project.zip env.py train.py requirements.txt Dockerfile
from google.colab import files
files.download("project.zip")

	zip warning: name not matched: train.py
  adding: env.py (deflated 52%)
  adding: requirements.txt (stored 0%)
  adding: Dockerfile (deflated 16%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install -q fastapi uvicorn requests

In [ ]:
%%writefile server.py
from fastapi import FastAPI
from env import LineWorld

app = FastAPI()
env = LineWorld()

@app.post("/reset")
def reset():
    obs, _ = env.reset()
    return {"obs": int(obs)}

@app.post("/step")
def step(action: int):
    obs, r, term, trunc, _ = env.step(action)
    return {"obs": int(obs), "reward": r, "done": term or trunc}

Writing server.py


In [ ]:
import subprocess, time
server = subprocess.Popen(["uvicorn", "server:app", "--port", "8000"])
time.sleep(3)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
import requests, random
URL = "http://localhost:8000"

obs = requests.post(f"{URL}/reset").json()["obs"]
done = False
while not done:
    res = requests.post(f"{URL}/step", params={"action": random.choice([0, 1])}).json()
    obs, done = res["obs"], res["done"]
print("finished at", obs)

server.terminate()  # clean up

finished at 9
